In [233]:
from pathlib import Path
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values

In [ ]:

DATA_DIR = Path(
    r"data"
)

df_clase1 = pd.read_csv(DATA_DIR / "clase_1.csv", sep=";")
df_clase2 = pd.read_csv(DATA_DIR / "clase_2.csv", sep=";")
df_clase3 = pd.read_csv(DATA_DIR / "clase_3.csv", sep=";")
df_clase4 = pd.read_csv(DATA_DIR / "clase_4.csv", sep=";")
df_claustro = pd.read_csv(DATA_DIR / "claustro.csv", sep=";")

df_clase1.head()

In [235]:
#df_clase1.info()

In [ ]:
df_clase2.head()

In [237]:
#df_clase2.info()

In [ ]:
df_clase3.head()

In [239]:
#df_clase3.info()

In [ ]:
df_clase4.head()

In [241]:
#df_clase4.info()

In [ ]:
df_claustro.head()

In [243]:
#df_claustro.info()

----------------------------------------------------------------------------------------------------------------


In [ ]:

df_clase1["vertical"] = "DS"
df_clase2["vertical"] = "DS"
df_clase3["vertical"] = "FS"
df_clase4["vertical"] = "FS"

df_alumnos_total = (pd.concat([df_clase1, df_clase2, df_clase3, df_clase4], ignore_index=True).rename(columns={"Promoción": "Promocion", "Proyecto_FullSatck": "Proyecto_FullStack"}))

df_alumno = df_alumnos_total[["Nombre", "Email", "Promocion"]].copy()
df_alumno.insert(0, "id_alumno", range(1, len(df_alumno) + 1))

df_alumno


In [ ]:
df_profesor = df_claustro[["Nombre", "Modalidad", "Campus"]].copy()
df_profesor.insert(0, "id_profesor", range(1, len(df_profesor)+1))
df_profesor

In [ ]:

df_vertical = pd.DataFrame({"Vertical": ["DS", "FS"]})
df_vertical.insert(0, "id_vertical", range(1, len(df_vertical) + 1))
df_vertical



In [ ]:

df_campus = pd.DataFrame({"Campus": ["Madrid", "Valencia"]})
df_campus.insert(0, "id_campus", range(1, len(df_campus) + 1))

df_campus

In [ ]:
mapa_campus = dict(zip(df_campus["Campus"], df_campus["id_campus"]))
mapa_vertical = dict(zip(df_vertical["Vertical"], df_vertical["id_vertical"]))

df_promocion_total = (
    df_alumnos_total[["Promocion", "Fecha_comienzo", "Campus", "vertical"]]
    .drop_duplicates()
    .reset_index(drop=True)
    .copy()
)

# Creamos la PK
df_promocion_total.insert(0, "id_promocion", range(1, len(df_promocion_total) + 1))

# Preparamos las FK
df_promocion_total["id_campus"] = df_promocion_total["Campus"].map(mapa_campus)
df_promocion_total["id_vertical"] = df_promocion_total["vertical"].map(mapa_vertical)

# Convertimos la fecha
df_promocion_total["Fecha_comienzo"] = pd.to_datetime(
    df_promocion_total["Fecha_comienzo"],
    dayfirst=True
).dt.date

# Seleccionamos las columnas definitivas
df_promocion_total = df_promocion_total[
    ["id_promocion", "id_campus", "id_vertical", "Promocion", "Fecha_comienzo"]
]

df_promocion_total

In [249]:
#df_alumnos_total.columns.tolist()

In [ ]:
"""
print("Dimensiones:", df_promocion_total.shape)
print("Nulos:", df_promocion_total.isnull().sum().sum())
print("PK única:", df_promocion_total["id_promocion"].is_unique)

print("FK campus válida:",
      df_promocion_total["id_campus"].isin(df_campus["id_campus"]).all())

print("FK vertical válida:",
      df_promocion_total["id_vertical"].isin(df_vertical["id_vertical"]).all())
"""

In [ ]:
# Recuperamos la información original de cada alumno
df_alumno_aux = df_alumnos_total[
    ["Email", "Promocion", "Fecha_comienzo", "Campus", "vertical"]
].copy()

# Recuperamos los identificadores
df_alumno_aux["id_campus"] = df_alumno_aux["Campus"].map(mapa_campus)
df_alumno_aux["id_vertical"] = df_alumno_aux["vertical"].map(mapa_vertical)

# Convertimos la fecha al mismo formato que en promociones
df_alumno_aux["Fecha_comienzo"] = pd.to_datetime(
    df_alumno_aux["Fecha_comienzo"],
    dayfirst=True
).dt.date

# Recuperamos el identificador de promoción
df_alumno_aux = df_alumno_aux.merge(
    df_promocion_total,
    on=["Promocion", "Fecha_comienzo", "id_campus", "id_vertical"],
    how="left",
    validate="many_to_one"
)

# Incorporamos id_promocion a la tabla alumno
df_alumno = df_alumno[["id_alumno", "Nombre", "Email"]].merge(
    df_alumno_aux[["Email", "id_promocion"]],
    on="Email",
    how="left",
    validate="one_to_one"
)

df_alumno

In [ ]:
"""
print("Dimensiones:", df_alumno.shape)
print("Nulos:", df_alumno.isnull().sum().sum())
print("PK única:", df_alumno["id_alumno"].is_unique)

print("FK promoción válida:",
      df_alumno["id_promocion"].isin(
          df_promocion_total["id_promocion"]
      ).all())

print("FK evaluaciones válida:",
      df_evaluacion["id_alumno"].isin(
          df_alumno["id_alumno"]
      ).all())

print(df_alumno.groupby("id_promocion").size())"""

----------------------------------------------------------------------------------------------

In [ ]:
mapa_campus = dict(zip(df_campus["Campus"], df_campus["id_campus"]))
mapa_vertical = dict(zip(df_vertical["Vertical"], df_vertical["id_vertical"]))

df_promocion_total = (
    df_alumnos_total[["Promocion", "Fecha_comienzo", "Campus", "vertical"]]
    .drop_duplicates()
    .reset_index(drop=True)
    .copy()
)

# Creamos la PK
df_promocion_total.insert(0, "id_promocion", range(1, len(df_promocion_total) + 1))

# Preparamos las FK
df_promocion_total["id_campus"] = df_promocion_total["Campus"].map(mapa_campus)
df_promocion_total["id_vertical"] = df_promocion_total["vertical"].map(mapa_vertical)

# Convertimos la fecha
df_promocion_total["Fecha_comienzo"] = pd.to_datetime(
    df_promocion_total["Fecha_comienzo"],
    dayfirst=True
).dt.date

# Seleccionamos las columnas definitivas
df_promocion_total = df_promocion_total[
    ["id_promocion", "id_campus", "id_vertical", "Promocion", "Fecha_comienzo"]
]

df_promocion_total

In [ ]:
df_modalidad = pd.DataFrame({"Modalidad": ["Presencial", "Online"]})
df_modalidad.insert(0, "id_modalidad", range(1, len(df_modalidad) + 1))

df_modalidad

In [ ]:
# Creamos el diccionario de correspondencia
mapa_modalidad = dict(zip(
    df_modalidad["Modalidad"],
    df_modalidad["id_modalidad"]
))

# Incorporamos el identificador de modalidad
df_profesor["id_modalidad"] = df_profesor["Modalidad"].map(mapa_modalidad)
df_profesor["id_campus"] = df_profesor["Campus"].map(mapa_campus)

# Seleccionamos las columnas definitivas
df_profesor = df_profesor[
    ["id_profesor", "Nombre", "id_modalidad", "id_campus"]
].copy()

df_profesor

In [ ]:
"""
print("Dimensiones:", df_profesor.shape)
print("Nulos:", df_profesor.isnull().sum().sum())
print("PK única:", df_profesor["id_profesor"].is_unique)

print("FK modalidad válida:",
      df_profesor["id_modalidad"].isin(
          df_modalidad["id_modalidad"]
      ).all())

print(df_profesor.groupby("id_modalidad").size())
"""

In [ ]:
df_proyec = pd.DataFrame({"Proyecto": ["Proyecto_HLF", "Proyecto_EDA", "Proyecto_BBDD","Proyecto_ML","Proyecto_Deployment","Proyecto_WebDev","Proyecto_FrontEnd","Proyecto_Backend","Proyecto_React","Proyecto_FullStack"]})
df_proyecto = pd.concat([df_vertical["id_vertical"], df_proyec], axis=1)
df_proyecto.insert(0, "id_proyecto", range(1, len(df_proyecto) + 1))

df_proyecto["id_vertical"] = df_proyecto["id_proyecto"].map({
    1: 1,
    2: 1,
    3: 1,
    4: 1,
    5: 1,
    6: 2,
    7: 2,
    8: 2,
    9: 2,
    10: 2
})

df_proyecto



In [ ]:


df_ds = pd.concat([df_clase1, df_clase2], ignore_index=True)
df_fs = pd.concat([df_clase3, df_clase4], ignore_index=True)


evaluacion_ds = df_ds.melt(id_vars=["Email"], value_vars=[c for c in df_ds.columns if c.startswith("Proyecto_")],
var_name="Proyecto",
value_name="Nota")

evaluacion_fs = df_fs.melt(id_vars=["Email"], value_vars=[c for c in df_fs.columns if c.startswith("Proyecto_")],
var_name="Proyecto",
value_name="Nota")

df_evaluacion = pd.concat([evaluacion_ds, evaluacion_fs], ignore_index=True)

df_evaluacion["Proyecto"] = df_evaluacion["Proyecto"].replace("Proyecto_FullSatck", "Proyecto_FullStack")

df_evaluacion = df_evaluacion.merge(df_alumno[["Email", "id_alumno"]], on="Email", how="left", validate="many_to_one")

df_evaluacion = df_evaluacion.merge(df_proyecto[["Proyecto", "id_proyecto"]], on="Proyecto", how="left", validate="many_to_one")

df_evaluacion = df_evaluacion[["id_proyecto", "id_alumno", "Nota"]].copy()

df_evaluacion.insert(0, "id_evaluacion", range(1, len(df_evaluacion) + 1))


df_evaluacion

In [259]:
df_alumno.to_csv("alumno.csv", index=False)
df_profesor.to_csv("profesor.csv", index=False)
df_vertical.to_csv("vertical.csv", index=False)
df_campus.to_csv("campus.csv", index=False)
df_promocion_total.to_csv("promocion.csv", index=False)
df_modalidad.to_csv("modalidad.csv", index=False)
df_proyecto.to_csv("proyecto.csv", index=False)
df_evaluacion.to_csv("evaluacion.csv", index=False)


In [260]:
#Funciones


def insertar_csv(cur, csv_dir: Path, csv_name: str, tabla: str, columnas: list[str]) -> None:
    ruta = csv_dir / csv_name
    df = pd.read_csv(ruta, encoding="utf-8")
    df.columns = [c.lower() for c in df.columns]

    assert df[columnas].isna().sum().sum() == 0, f"Hay valores nulos en {csv_name} para '{tabla}'"

    filas = [tuple(row) for row in df[columnas].itertuples(index=False, name=None)]
    cols_sql = ", ".join(columnas)
    query = f"INSERT INTO {tabla} ({cols_sql}) VALUES %s;"

    execute_values(cur, query, filas)
    print(f"OK -> '{tabla}': {len(filas)} filas insertadas.")


def sincronizar_secuencias(cur, secuencias: list[tuple[str, str]]) -> None:
    for tabla, pk in secuencias:
        cur.execute(
            f"""
            SELECT setval(
                pg_get_serial_sequence('{tabla}', '{pk}'),
                COALESCE((SELECT MAX({pk}) FROM {tabla}), 1),
                true
            );
            """
        )

In [ ]:


import psycopg2


def main() -> None:
    csv_dir = Path(".") 

    conn = psycopg2.connect(
        host="localhost",
        port=5432,
        dbname="Base de datos 2 grupo",
        user="postgres",
        password="Inaki",
    )

    secuencias = [
        ("vertical", "id_vertical"),
        ("campus", "id_campus"),
        ("modalidad", "id_modalidad"),
        ("promocion", "id_promocion"),
        ("profesor", "id_profesor"),
        ("proyecto", "id_proyecto"),
        ("alumno", "id_alumno"),
        ("evaluacion", "id_evaluacion"),
    ]

    try:
        with conn:
            with conn.cursor() as cur:
                insertar_csv(cur, csv_dir, "vertical.csv", "vertical", ["id_vertical", "vertical"])
                insertar_csv(cur, csv_dir, "campus.csv", "campus", ["id_campus", "campus"])
                insertar_csv(cur, csv_dir, "modalidad.csv", "modalidad", ["id_modalidad", "modalidad"])

                insertar_csv(
                    cur,
                    csv_dir,
                    "promocion.csv",
                    "promocion",
                    ["id_promocion", "id_campus", "id_vertical", "promocion", "fecha_comienzo"],
                )
                insertar_csv(
                    cur,
                    csv_dir,
                    "profesor.csv",
                    "profesor",
                    ["id_profesor", "nombre", "id_modalidad", "id_campus"],
                )
                insertar_csv(cur, csv_dir, "proyecto.csv", "proyecto", ["id_proyecto", "id_vertical", "proyecto"])

                insertar_csv(cur, csv_dir, "alumno.csv", "alumno", ["id_alumno", "nombre", "email", "id_promocion"])
                insertar_csv(
                    cur,
                    csv_dir,
                    "evaluacion.csv",
                    "evaluacion",
                    ["id_evaluacion", "id_proyecto", "id_alumno", "nota"],
                )

                sincronizar_secuencias(cur, secuencias)

        print("Carga completa en PostgreSQL.")

    except Exception as e:
        conn.rollback()
        print("Error en la carga; se ha hecho ROLLBACK:", e)
        raise

    finally:
        conn.close()


main()